<a href="https://colab.research.google.com/github/Mhmd-Seyed/ecommerce-behavior-analytics/blob/main/03_spark_multidimensional_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder \
    .appName("Student3_Multidimensional_Analysis") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()
# Test için sample_events.parquet, final çalıştırmada events_clean.parquet kullanın
input_path = "/content/drive/MyDrive/ecommerce_project/events_clean.parquet"
df = spark.read.parquet(input_path)
df.cache()
# Spark SQL sorguları yazabilmek için geçici tablo görünümü oluştur
df.createOrReplaceTempView("events")
print("Analize hazır toplam olay sayısı:", df.count())

In [ ]:
# 1. BOYUT: ÜRÜN BAZLI ANALİZ (En çok etkileşim alan ve ciro üreten ilk 25 ürün)
top_products_df = spark.sql("""
    SELECT
        product_id,
        FIRST(main_category) AS category,
        FIRST(brand) AS brand,
        ROUND(AVG(price), 2) AS avg_price,
        SUM(is_view) AS total_views,
        SUM(is_cart) AS total_carts,
        SUM(is_purchase) AS total_purchases,
        ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 2) AS
total_revenue,
        ROUND((SUM(is_purchase) / NULLIF(SUM(is_view), 0)) * 100, 2) AS
view_to_purchase_pct
    FROM events
    GROUP BY product_id
    HAVING total_views >= 50
    ORDER BY total_revenue DESC
    LIMIT 25
""")
print("=== EN YÜKSEK CİRO ÜRETEN ÜRÜNLER ===")
top_products_df.show(10, truncate=False)
# Yüksek görüntülenmeye rağmen satışı düşük kalan sorunlu/fırsat ürünleri tespit et
low_conv_products_df = spark.sql("""
    SELECT
        product_id,
        FIRST(main_category) AS category,
        ROUND(AVG(price), 2) AS avg_price,
        SUM(is_view) AS total_views,
        SUM(is_purchase) AS total_purchases,
        ROUND((SUM(is_purchase) / NULLIF(SUM(is_view), 0)) * 100, 2) AS
view_to_purchase_pct
    FROM events
    GROUP BY product_id
    HAVING total_views >= 500 AND view_to_purchase_pct < 0.5
    ORDER BY total_views DESC
    LIMIT 10
""")
print("=== ÇOK GÖRÜNTÜLENİP AZ SATILAN ÜRÜNLER ===")
low_conv_products_df.show(10, truncate=False)

In [ ]:
# 2. BOYUT: KATEGORİ BAZLI ANALİZ (Ana kategori kırılımında performans ve ciro)
category_df = spark.sql("""
    SELECT
        main_category,
        COUNT(DISTINCT product_id) AS unique_products,
        SUM(is_view) AS total_views,
        SUM(is_cart) AS total_carts,
        SUM(is_purchase) AS total_purchases,
        ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 2) AS total_revenue,
        ROUND((SUM(is_purchase) / NULLIF(SUM(is_view), 0)) * 100, 2) AS conversion_rate_pct
    FROM events
    GROUP BY main_category
    ORDER BY total_revenue DESC
""")

print("=== KATEGORİ BAZLI PERFORMANS VE CİRO ANALİZİ ===")
category_df.show(truncate=False)

In [ ]:
# 3. BOYUT: SAAT BAZINDA ZAMAN ANALİZİ (00:00 - 23:00 arası yoğunluk ve dönüşüm)
hourly_df = spark.sql("""
    SELECT
        event_hour,
        SUM(is_view) AS total_views,
        SUM(is_cart) AS total_carts,
        SUM(is_purchase) AS total_purchases,
        ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 2) AS
total_revenue,
        ROUND((SUM(is_purchase) / NULLIF(SUM(is_view), 0)) * 100, 2) AS
hourly_conversion_pct
    FROM events
    GROUP BY event_hour
    ORDER BY event_hour ASC
""")
print("=== SAAT BAZINDA DAVRANIŞ VE DÖNÜŞÜM ANALİZİ ===")
hourly_df.show(24, truncate=False)

In [ ]:
# 4. BOYUT: MÜŞTERİ SEGMENTİ BAZLI ANALİZ (Kullanıcı bazında davranışsal segmentasyon)
user_summary_df = spark.sql("""
    SELECT
        user_id,
        COUNT(DISTINCT user_session) AS total_sessions,
        SUM(is_view) AS user_views,
        SUM(is_cart) AS user_carts,
        SUM(is_purchase) AS user_purchases,
        ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 2) AS
user_total_spent
    FROM events
    GROUP BY user_id
""")
user_segmented_df = user_summary_df.withColumn(
    "customer_segment",
    F.when(F.col("user_purchases") >= 2, "4_Sadik_Tekrarli_Musteri")
     .when(F.col("user_purchases") == 1, "3_Tek_Seferlik_Alici")
     .when(F.col("user_carts") > 0, "2_Sepet_Terk_Eden")
     .otherwise("1_Sadece_Gezinen")
)
segment_report_df = user_segmented_df.groupBy("customer_segment").agg(
    F.count("user_id").alias("customer_count"),
    F.round(F.avg("total_sessions"), 2).alias("avg_sessions_per_user"),
    F.round(F.avg("user_views"), 2).alias("avg_views_per_user"),
    F.round(F.sum("user_total_spent"), 2).alias("segment_total_revenue"),
    F.round(F.avg(F.when(F.col("user_total_spent") > 0,
F.col("user_total_spent"))), 2).alias("avg_spending_per_buyer")
).orderBy("customer_segment")
print("=== MÜŞTERİ SEGMENTİ PERFORMANS RAPORU ===")
segment_report_df.show(truncate=False)

In [ ]:
output_dir = "/content/drive/MyDrive/ecommerce_project"
top_products_df.toPandas().to_csv(f"{output_dir}/top_products.csv", index=False)
category_df.toPandas().to_csv(f"{output_dir}/category_analysis.csv", index=False)
hourly_df.toPandas().to_csv(f"{output_dir}/hourly_trends.csv", index=False)
segment_report_df.toPandas().to_csv(f"{output_dir}/customer_segments.csv",
index=False)
df.unpersist()
print("4 adet çok boyutlu analiz CSV dosyası başarıyla kaydedildi!")